# Strict-Schema Collection (Shapefile-Derived)

End-to-end use case demonstrating the strict-schema building blocks shipped 2026-04-30:

| PR | Capability |
|---|---|
| #178 | `CollectionSchema.strict_unknown_fields` — refuse writes with undeclared fields (HTTP 422) |
| #179 | `CollectionSchema.materialize_fields_as_columns` — declared fields become native PG columns |
| #180 | Post-ingest `ANALYZE` — planner stats refreshed automatically after ingest |
| #181 | `extract_ogr_schema(uri)` — derive schema directly from a shapefile via OGR |

Plus pre-existing capabilities used here:
- `CollectionWritePolicy.on_conflict = REFUSE_FAIL` + `external_id_field = 'id'` — the same id cannot be written twice into the same collection (HTTP 409).
- RFC 9457 Problem Details on errors (PR #168).

## What this notebook does

This notebook runs in **JupyterLite (Pyodide)**, which has no GDAL/OGR bindings and no shell environment for tokens. So we:
1. Fetch a sysadmin token from Keycloak inline (fallback to `DYNASTORE_*_TOKEN` env vars when present).
2. Hardcode the `fields_payload` that `extract_ogr_schema()` would have produced for a small synthetic roads shapefile (geometry + 4 fields).
3. PATCH `CollectionSchema` (strict + materialize) and `CollectionWritePolicy` (refuse_fail).
4. Write features directly via `POST /features/.../items` (skipping the asset-upload + ingestion-task path, which requires a worker pod and OGR).
5. Demonstrate the rejection paths: rogue field → 422, duplicate id → 409.

The "production path" section at the end shows the equivalent server-side flow (asset upload → ingestion task → `extract_ogr_schema` on the worker) for non-Pyodide environments.

In [ ]:
import os
import json
import asyncio
import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
KEYCLOAK = "http://localhost:8180/realms/geoid/protocol/openid-connect/token"
CATALOG_ID = "demo-strict-schema"
COLLECTION_ID = "roads"

TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

if not TOKEN:
    print("No env token — fetching one from Keycloak (testadmin)…")
    async with httpx.AsyncClient(timeout=15.0) as kc:
        r = await kc.post(KEYCLOAK, data={
            "grant_type": "password",
            "client_id": "geoid-api",
            "client_secret": "geoid-api-secret",
            "username": "testadmin",
            "password": "testpassword",
        })
    if r.status_code == 200:
        TOKEN = r.json().get("access_token", "")
        print(f"Got token from Keycloak (len={len(TOKEN)})")
    else:
        print(f"Keycloak token request failed: {r.status_code} — {r.text[:200]}")

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.AsyncClient(base_url=BASE, headers=headers, timeout=120.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    body = ""
    try:
        body = r.text[:400]
    except Exception:
        pass
    print(f"{label + ': ' if label else ''}{status}{'' if ok else f' — {body}'}")
    return r


print(f"BASE  = {BASE}")
print(f"TOKEN = {'set (' + str(len(TOKEN)) + ' chars)' if TOKEN else 'EMPTY (writes will fail)'}")

## 1. The schema we would have derived from a roads shapefile

`extract_ogr_schema(uri)` (PR #181) introspects a shapefile via OGR and returns a `Dict[str, FieldDefinition]`. Pyodide has no OGR, so we hardcode what the call **would** have returned for a 3-feature roads dataset with fields `road_id` (str), `highway` (str), `lanes` (int), `surface` (str), plus a `LineString` geometry.

In [ ]:
# Equivalent to:
#   from dynastore.tasks.ingestion.schema_introspect import extract_ogr_schema
#   derived = extract_ogr_schema("/vsizip/<path>/roads.zip/roads.shp")
#   fields_payload = {n: fd.model_dump(exclude_none=True) for n, fd in derived.items()}
fields_payload = {
    "geometry": {"data_type": "geometry", "geometry_type": "linestring", "srid": 4326},
    "road_id":  {"data_type": "text"},
    "highway":  {"data_type": "text"},
    "lanes":    {"data_type": "integer"},
    "surface":  {"data_type": "text"},
}
for n, fd in fields_payload.items():
    print(f"  {n:10s} -> {fd}")

## 2. Create catalog + empty collection (clean slate)

Removes any leftover from a previous run, then creates a fresh catalog and an empty collection. The collection's schema gets PATCHed in step 3.

In [ ]:
_cleanup = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Strict-Schema Demo",
    "description": "Demonstrates strict + materialize + refuse_fail end-to-end.",
})
_check(r, "Catalog")

for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("WARN: catalog still provisioning after 30s")

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Roads",
    "description": "Synthetic roads dataset with strict OGR-derived schema.",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
})
_check(r, "Collection")

## 3. PATCH `CollectionSchema` — strict + materialize

- `strict_unknown_fields=True` (PR #178) — reject any feature whose properties are not in `fields`.
- `materialize_fields_as_columns=True` (PR #179) — lift declared fields into native PG columns (indexable, ANALYZE-able).

In [ ]:
patch_body = {
    "CollectionSchema": {
        "fields": fields_payload,
        "strict_unknown_fields": True,
        "materialize_fields_as_columns": True,
    },
}
r = await client.patch(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    json=patch_body,
)
_check(r, "PATCH CollectionSchema")

r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "true"},
)
data = r.json()
schema_block = (
    data.get("configs", {})
        .get("collection", {})
        .get("storage", {})
        .get("schema", {})
)
print("\nApplied CollectionSchema (excerpt):")
print(json.dumps(schema_block.get("CollectionSchema", schema_block), indent=2)[:600])

## 4. PATCH `CollectionWritePolicy` — refuse duplicate `id`

- `external_id_field = 'id'` — use the feature's `id` as the identity (always-valid field, no schema dependency).
- `on_conflict = refuse_fail` — raise `ConflictError` → HTTP 409 on a match.

In [ ]:
policy_body = {
    "CollectionWritePolicy": {
        "identity_matchers": ["external_id"],
        "external_id_field": "id",
        "on_conflict": "refuse_fail",
        "track_asset_id": True,
    },
}
r = await client.patch(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    json=policy_body,
)
_check(r, "PATCH CollectionWritePolicy")

## 5. Write 3 features directly

In Pyodide we cannot run the asset-upload + ingestion-task path (no OGR, no worker pod from the browser). Instead we POST the FeatureCollection directly — the same code path the ingestion task itself takes server-side.

In [ ]:
feature_collection = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "R001",
            "geometry": {"type": "LineString", "coordinates": [[0.0, 0.0], [0.1, 0.1]]},
            "properties": {"road_id": "R001", "highway": "primary",   "lanes": 2, "surface": "asphalt"},
        },
        {
            "type": "Feature",
            "id": "R002",
            "geometry": {"type": "LineString", "coordinates": [[0.2, 0.2], [0.3, 0.25]]},
            "properties": {"road_id": "R002", "highway": "secondary", "lanes": 1, "surface": "gravel"},
        },
        {
            "type": "Feature",
            "id": "R003",
            "geometry": {"type": "LineString", "coordinates": [[0.4, 0.3], [0.5, 0.4]]},
            "properties": {"road_id": "R003", "highway": "tertiary",  "lanes": 1, "surface": "dirt"},
        },
    ],
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=feature_collection,
)
_check(r, "Write 3 features")
if r.status_code < 400:
    print(json.dumps(r.json(), indent=2)[:400])

## 6. Strict-mode rejects features with rogue fields

A feature carrying `unauthorized_field` (not in `CollectionSchema.fields`) is refused with HTTP 422 + RFC 9457 Problem Details listing the offending field.

In [ ]:
rogue_feature = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "id": "rogue1",
        "geometry": {"type": "Point", "coordinates": [0.6, 0.5]},
        "properties": {
            "road_id": "X1",
            "highway": "primary",
            "lanes": 2,
            "surface": "asphalt",
            "unauthorized_field": "BOOM",
        },
    }],
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=rogue_feature,
)
print(f"Status: {r.status_code} (expected 422)")
print("Body:")
try:
    print(json.dumps(r.json(), indent=2)[:800])
except Exception:
    print(r.text[:800])

## 7. `on_conflict=refuse_fail` rejects duplicate `id`

A feature whose `id` collides with one already written is refused with HTTP 409 ConflictError.

In [ ]:
duplicate_feature = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "id": "R001",
        "geometry": {"type": "LineString", "coordinates": [[0.0, 0.0], [0.1, 0.1]]},
        "properties": {
            "road_id": "R001",
            "highway": "primary",
            "lanes": 4,
            "surface": "asphalt",
        },
    }],
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=duplicate_feature,
)
print(f"Status: {r.status_code} (expected 409)")
print("Body:")
try:
    print(json.dumps(r.json(), indent=2)[:800])
except Exception:
    print(r.text[:800])

## 8. Verify materialised columns + ingested rows

`/queryables` reflects the OGC-CRS-aware view of the schema; declared fields appear as native PG columns. `/items` confirms the 3 features are in storage.

In [ ]:
r = await client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables")
_check(r, "Queryables")
props = r.json().get("properties", {})
print("\nDeclared queryable properties:")
for name, schema in props.items():
    t = schema.get("type")
    print(f"  {name:18s} type={t}")

print("\nIngested items:")
r = await client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
items = r.json().get("features", [])
print(f"  count = {len(items)} (expected 3)")
for f in items:
    p = f.get("properties", {})
    print(f"  id={f.get('id')} road_id={p.get('road_id')} lanes={p.get('lanes')} surface={p.get('surface')}")

## Cleanup

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
_check(r, "DELETE catalog")
await client.aclose()
print("Done.")

## Production path (server-side, with OGR)

Outside Pyodide — from a CLI, a worker, or a Jupyter kernel that has GDAL installed — the full pipeline replaces steps 1 and 5 with the asset-upload + ingestion-task flow:

```python
# Step 1 (server-side): derive schema from the actual shapefile
from dynastore.tasks.ingestion.schema_introspect import extract_ogr_schema
derived = extract_ogr_schema(f"/vsizip/{zip_path}/roads.shp")
fields_payload = {n: fd.model_dump(exclude_none=True) for n, fd in derived.items()}

# Steps 3 + 4: same PATCHes as the notebook above.

# Step 5 (server-side): upload the shapefile asset, then run the ingestion task
init = (await client.post(
    f"/assets/{CATALOG_ID}/{COLLECTION_ID}/init-upload",
    json={"asset_id": ASSET_ID, "filename": "roads.zip", "content_type": "application/zip"},
)).json()
await upload_client.put(init["upload_url"], content=open(zip_path, "rb").read())
await client.post(f"/assets/{CATALOG_ID}/{COLLECTION_ID}/finalize/{ASSET_ID}", json={})

await client.post("/processes/ingestion/execution", json={
    "inputs": {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {"assets": [{"asset_id": ASSET_ID}]},
    },
})
```

The ingestion task runs `extract_ogr_schema()` against the asset's URI on the worker pod, opens the shapefile via `/vsizip/`, and writes features through the same `POST /features/.../items` code path used in step 5 of this notebook — so the strict-schema and conflict-rejection guarantees are identical.